In [61]:
!pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable


In [62]:
import pandas as pd
import numpy as np
import sklearn.feature_extraction.text as text
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = text.TfidfVectorizer()

In [63]:
dataset = pd.read_csv('../data/amazon_products.csv',nrows=10000)
print('=== Dataset Loaded ===')
print(f"Shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")

=== Dataset Loaded ===
Shape: (10000, 11)
Columns: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']


In [64]:
# creating the product descrption from the columns
def create_product_text(row):
    """Combine product features into one text string"""
    features = []
    # Adding title
    features.append(str(row['title']))

    # Adding Category
    features.append(f"category_{row['category_id']}")

    # Add price range (helps find similarly priced products)
    if row['price'] > 0:
        if row['price'] < 50:
            features.append("budget")
        elif row['price'] < 200:
            features.append("mid_range")
        else:
            features.append("Premium")

    
    # Adding Bestseller info
    if(row['isBestSeller']):
        features.append("bestSeller")

    return " ".join(features)

# Applying to all the products
dataset['product_text'] = dataset.apply(create_product_text, axis=1)
print('Product features Created')
print("\nExample:")
print(dataset['product_text'].iloc[0])


Product features Created

Example:
Sion Softside Expandable Roller Luggage, Black, Checked-Large 29-Inch category_104 mid_range


In [65]:
# Converting text to numerical numbers
vectorizer = text.TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(dataset['product_text'])
print(f"TF-IDF matrix created")
print(f"Shape: {tfidf_matrix.shape}")
print(f"Each Product is now a {tfidf_matrix.shape[1]}-dimensional vector")

TF-IDF matrix created
Shape: (10000, 5000)
Each Product is now a 5000-dimensional vector


In [66]:
# Calualating the cosine similarities between all the products
cosine_sim = cosine_similarity(tfidf_matrix,tfidf_matrix)
print(f"Similarity matrix created")
print(f"Shape: {cosine_sim.shape}")
print(f"Similarity between  prosuct 0 and 1: {cosine_sim[0][1]:.3f}")

Similarity matrix created
Shape: (10000, 10000)
Similarity between  prosuct 0 and 1: 0.154


In [67]:
# Build the Recommender
def recommend_similar_products(product_title, n_recommendations = 5):
    """
    Find products similar to the given product
    
    Parameters:
    - product_title: Name of the product
    - n_recommendations: Number of recommendations to return
    
    Returns:
    - DataFrame with recommended products
    """
    # Find the product in our dataset
    matching_products = dataset[dataset['title'].str.contains(product_title, case=False)]
    
    if len(matching_products) == 0:
        return f"Product '{product_title}' not Found"
    # Get the first matching product's index
    product_idx = matching_products.index[0]

    # Get similarity scores for this product
    similarity_scores = list(enumerate(cosine_sim[product_idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similar_indices = [i[0] for i in similarity_scores[1:n_recommendations+1]]

    recommendations = dataset.iloc[similar_indices][['title', 'price', 'stars', 'boughtInLastMonth']]
    recommendations['similarity_score'] = [similarity_scores[i+1][1] for i in range(n_recommendations)]

    return recommendations

print("Testing Content-based Recommender\n ")
print("="*60)

Testing Content-based Recommender
 


In [68]:
# Testing the model
print("Test 1: Products similar to 'Luggage'\n")
print(recommend_similar_products("Luggage", 5))
print("\n" + "="*60 + "\n")

Test 1: Products similar to 'Luggage'

                                                 title   price  stars  \
17   Ascella X Softside Expandable Luggage with Spi...  198.41    4.5   
494  28 Inch Luggage with Spinner Wheels Softside E...  149.99    0.0   
319  Cambridge Lightweight Luggage Softside Expanda...  164.99    4.3   
380  Bromley Softside Expandable Spinner Luggage, B...  172.19    4.5   
24   Anzio Softside Expandable Spinner Luggage, Tea...   98.62    4.1   

     boughtInLastMonth  similarity_score  
17                 100          0.576787  
494                  0          0.460594  
319                  0          0.445873  
380                  0          0.430355  
24                 300          0.420140  




In [69]:
print("Test 2: Products similar to 'Phone'\n")
print(recommend_similar_products("Phone", 5))
print("\n" + "="*60 + "\n")

Test 2: Products similar to 'Phone'

                                                  title   price  stars  \
893   Travel Painting Travel Luggage Cover Suitcase ...   36.49    0.0   
215   Carry On Luggage 20'' Travel Suitcase Rolling ...  169.99    4.8   
59    Large Luggage Suitcase with Wheels, Check-in 2...  129.99    4.8   
293   Luggage Suitcase with Spinner Wheels, 24'' Che...  199.99    4.4   
3097  Sports Baseball Sliding Shorts with Cup Holder...   16.99    4.3   

      boughtInLastMonth  similarity_score  
893                   0          0.296508  
215                   0          0.289847  
59                   50          0.259022  
293                   0          0.257343  
3097                200          0.248621  




In [70]:
sample_product = dataset['title'].iloc[100]
print(f"Test 3: Products similar to '{sample_product}'\n")
print(recommend_similar_products(sample_product[:30], 5))

Test 3: Products similar to 'Carry On Luggage Airline Approved, Softside Suitcase with Spinner Wheels 20 Inch Small Lightweight Carry-on Luggage with Front Pockets, Women Men Business Travel (Black)'

                                                 title   price  stars  \
100  Carry On Luggage Airline Approved, Softside Su...   85.99    4.4   
142  Luggage Hard Shell Suitcases Airline Approved ...  189.99    4.2   
45   Carry On Luggage 22x14x9 Airline Approved, 1OO...   84.79    4.3   
87   20'' Carry on Luggage Lightweight with Spinner...   59.99    4.6   
441  28" Checked Luggage, Airline Approved Spinner ...  129.99    0.0   

     boughtInLastMonth  similarity_score  
100                 50          0.706256  
142                  0          0.614664  
45                 100          0.605384  
87                 100          0.603347  
441                  0          0.597596  


In [71]:
def evaluate_recommendation(product_title):
    """See how well our recommender works"""
    
    print(f" Analyzing recommendations for: {product_title}\n")
    
    recs = recommend_similar_products(product_title, 5)
    
    if isinstance(recs, str):
        print(recs)
        return
    
    print("RECOMMENDATIONS:")
    for i, row in recs.iterrows():
        print(f"{i+1}. {row['title'][:60]}...")
        print(f" Price: ${row['price']} | Stars: {row['stars']}")
        print(f" Similarity: {row['similarity_score']:.3f}")
        print()
    
    # Check if recommendations make sense
    avg_price = recs['price'].mean()
    avg_stars = recs['stars'].mean()
    
    print(f" Summary:")
    print(f" Average price of recommendations: ${avg_price:.2f}")
    print(f" Average rating: {avg_stars:.2f}")

# Test evaluation
evaluate_recommendation("Suitcase")

 Analyzing recommendations for: Suitcase

RECOMMENDATIONS:
86. Luggage PC+ABS Durable Expandable Hardside Suitcase with Dou...
 Price: $129.99 | Stars: 4.6
 Similarity: 0.852

10. Luggage Sets Expandable Lightweight Suitcases with Wheels PC...
 Price: $209.99 | Stars: 4.4
 Similarity: 0.729

140. Luggage Expandable PC ABS Durable Hardside Suitcase with Spi...
 Price: $139.99 | Stars: 4.6
 Similarity: 0.644

33. Luggage Sets 3 Piece Softside Expandable Lightweight Durable...
 Price: $151.99 | Stars: 4.5
 Similarity: 0.643

73. Luggage Sets Expandable ABS Hardshell 3pcs Clearance Luggage...
 Price: $179.99 | Stars: 4.5
 Similarity: 0.640

 Summary:
 Average price of recommendations: $162.39
 Average rating: 4.52


In [72]:
# Save the model artifacts for later use
import os
import pickle
# Create the directory if it doesn't exist
os.makedirs('../artifacts', exist_ok=True)

# Now your existing code will work
with open('../artifacts/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)